In [1]:
from typing_extensions import Self
from typing import List, Dict, Set, Tuple
import math

class CyclicPermutation:
    def __init__(self, groups: List[List[int]]):
        for group in groups:
            assert len(set(group)) == len(group), f"Must be unique number, got {group}"
        
        self.groups = groups

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}({self.groups})"

    @staticmethod
    def _simulate(groups: List[List[int]], boxes: Dict):
        for group in groups:
            if not group:
                continue

            if group[0] not in boxes:
                boxes[group[0]] = group[0]
            cache = boxes[group[0]]
            for i in range(1, len(group) + 1):
                i %= len(group)
                # init
                if group[i] not in boxes:
                    boxes[group[i]] = group[i]

                boxes[group[i]], cache = cache, boxes[group[i]]
        return boxes

    @staticmethod
    def _build_one_group(start_k:int, boxes: Dict, visited:Set):
        group = [start_k]
        visited.add(start_k)
                
        while True:
            v = boxes[start_k]
            start_k = v
            visited.add(start_k)

            if start_k in group:
                group = group[::-1]
                return group

            # append to the first
            group.append(start_k)

    @staticmethod 
    def _build_groups(boxes: Dict):
        visited = set()
        groups = []
        for v in boxes.values():
            if v not in visited:
                group = CyclicPermutation._build_one_group(v, boxes, visited)
                groups.append(group)

        return groups

    def __add__(self, other:Self):
        boxes = {}
        self._simulate(self.groups, boxes)
        other._simulate(other.groups, boxes)

        # build
        groups = self._build_groups(boxes)

        return CyclicPermutation(groups)
    
    def __sub__(self, other:Self):
        return self + other.inv()
    
    def __mul__(self, other: int):
        res = CyclicPermutation([[]])
        for _ in range(other):
            res += self
        return res
    
    def __eq__(self, value: object) -> bool:
        if not isinstance(value, CyclicPermutation):
            raise ValueError(f"Can only compare with {self.__class__.__name__}. Got {value}")
        # same cp has the same inverse
        return (self.inv() + value).is_identity()

    def is_identity(self) -> bool:
        return math.lcm(*(len(g) for g in self.groups)) == 1

    def inv(self):
        lcm = math.lcm(*(len(g) for g in self.groups))
        return self * (lcm - 1)

In [2]:
cp = CyclicPermutation([[1, 2], [2, 3]]) + CyclicPermutation([[3, 4], [4,5]])
cp * 0, cp * 1, cp* 2, cp*3, cp*4, cp*5, cp*6

(CyclicPermutation([[]]),
 CyclicPermutation([[1, 5, 4, 3, 2]]),
 CyclicPermutation([[1, 4, 2, 5, 3]]),
 CyclicPermutation([[1, 3, 5, 2, 4]]),
 CyclicPermutation([[1, 2, 3, 4, 5]]),
 CyclicPermutation([[1], [2], [3], [4], [5]]),
 CyclicPermutation([[1, 5, 4, 3, 2]]))

In [3]:
def coor_to_int(size_n, y, x) -> int:
    x += 1
    return y * size_n  + x

def int_to_coor(size_n, i) -> Tuple[int, int]:
    """ return y, x """
    return i // size_n, i % size_n

for i in range(9):
    y, x= int_to_coor(3, i)
    inch = coor_to_int(3, y, x)
    print(y, x, inch)

0 0 1
0 1 2
0 2 3
1 0 4
1 1 5
1 2 6
2 0 7
2 1 8
2 2 9


In [4]:
def build_cp_from_tables(arr: List[List[int]]) -> Tuple[CyclicPermutation, int]:
    start_0 = 0
    boxes = dict()
    n_size = len(arr)
    for y in range(n_size):
        for x in range(n_size):
            # NOTE: we must keep original, like the correct state key: value must has key = value
            # like 1: 1, 2: 2, 3:3, 0: 0
            i = coor_to_int(n_size, y, x)

            if y + 1 == n_size and x + 1 == n_size:
                i = 0

            boxes[i] = arr[y][x]

            # side tak: keep track of your start
            if arr[y][x] == 0:
                start_0 = i


    # build cyclic permutation
    groups = CyclicPermutation._build_groups(boxes)

    return CyclicPermutation(groups), start_0

build_cp_from_tables([[3,2,1],
                      [4,5,6],
                      [7,8,0]])

(CyclicPermutation([[1, 3], [2], [4], [5], [6], [7], [8], [0]]), 0)

In [5]:
a = build_cp_from_tables([[1, 3],
                      [2, 0]])[0]
a

CyclicPermutation([[1], [2, 3], [0]])

In [151]:
def _is_valid_steps(n_size:int, origin:int, comp:int, n_size2 = None):
    if origin == 0:
        origin = n_size * n_size
    if comp == 0:
        comp = n_size * n_size

    # left case, only exception of on last of previous line
    if comp == origin - 1:
        if comp % n_size == 0:
            return False
        return True

    # right case, only excepton of on first  of next line    
    if comp == origin + 1:
        if comp % n_size == 1:
            return False
        return True
        
    # top case, with only exception of smaller than 1
    if comp == origin - n_size:
        if comp < 1:
            return False    
        return True
    
    # under case, with only exceptoin of larger than  n_size ^ 2
    if comp == origin + n_size:
        # for optimizing only, because for one table, we will check many
        # so we should skip the re-calculating the square for the whole list
        if n_size2 is None:
            n_size2 = n_size ** 2
        
        if comp > n_size2:
            return False
        return True
    
    # it is in somewhere else
    return False

def _validate_steps(steps: List[int], n_size: int):
    for s1, s2 in zip(steps[:-1], steps[1:]):
        if s1 == s2:
            raise ValueError(f"Why swap with oneself? Got {s1}")
        
        if not _is_valid_steps(n_size, s1, s2, n_size2=n_size**2):
            raise ValueError(f"Not valid sliding move. Got {s1} -> {s2}")
        
    return True

def go(next_steps:List[int], start:int,*, n_size: int = 2, check_valid: bool = True) -> CyclicPermutation:
    if check_valid:
        _validate_steps(next_steps, n_size)
        _validate_steps([start, next_steps[0]], n_size)

    start_cache = start
    cp = CyclicPermutation([[]])
    for step in next_steps:
        cp += CyclicPermutation([[start_cache, step]])
        start_cache = step

    return cp

def _yield_valid_steps(n_size:int, origin:int, n_size2 = None):
    if origin == 0:
        origin = n_size * n_size

    if n_size2 is None:
        n_size2 = n_size ** 2
   

    # left case, only exception of on last of previous line
    comp = origin - 1
    if comp % n_size == 0:
        pass
    else:
        yield comp % n_size2

    # right case, only excepton of on first  of next line    
    comp = origin + 1
    if comp % n_size == 1:
        pass
    else:
        yield comp % n_size2
        
    # top case, with only exception of smaller than 1
    comp = origin - n_size
    if comp < 1:
        pass
    else:
        yield comp % n_size2
    
    # under case, with only exceptoin of larger than  n_size ^ 2
    comp = origin + n_size
        # for optimizing only, because for one table, we will check many
        # so we should skip the re-calculating the square for the whole list
    
    if comp > n_size2:
        pass
    else:
        yield comp % n_size2

In [154]:
# 1 2 3
# 4 5 6
# 7 8 9
[*_yield_valid_steps(3, 8)]

[7, 0, 5]

In [ ]:
def solve22(inv:CyclicPermutation, start:int, n_size:int, visited: Set) -> List[int]:
    n_size2 = n_size ** 2
    index = -1
    steps= []
    found=  False
    for g in inv.groups:
        if len(g) == 1:
            continue
        
        for i in range(len(g)):
            if _is_valid_steps(n_size, start, g[i], n_size2=n_size2):
                if g[i] not in visited:
                    index = i
                    visited.add(g[i])
                    found = True
                    break
        if found:
            steps = g[index::-1] + g[-1:index - len(g):-1]
            if steps[-1] != start:
                return steps[:-1]
            return steps
    
    return None
    

In [112]:
def test(table):
    n_size = len(table)
    cp_prob, start = build_cp_from_tables(table)
    inv = cp_prob.inv()
    print("\n=== Start Position ===")
    print(start)
    print("\n=== Inverse ===")
    print(inv)
    visited = set()
    while True:
        try:
            steps = solve22(inv, start, n_size, visited)
            if steps is None:
                print("\n--- Visited States ---")
                print(visited)
                print("======================\n")
                break
            
            print("\n>>> Steps Found:")
            print(steps)
            sol = go(steps, start, n_size=n_size)
            print("\n>>> Solution:")
            print(sol)
            print("\n>>> Matches Inverse?:", sol == inv)
            print("----------------------\n")
        except Exception as e:
            print("\n!!! Exception Caught !!!")
            print(e)
            print("========================\n")


def sramble(n_size: int):
    import random
    nsize2 = n_size**2
    table= []
    base_row = [*range(1, n_size + 1)]
    for i in range(n_size):
        row = [i * n_size + br for br in base_row]
        if row[-1] == nsize2:
            row[-1] = 0
        table.append(row)
    
    st = [random.randint(0, n_size ** 2 - 1) for _ in range(20)]
    scrable_steps = []
    cache_step = 0
    for s in st:
        if _is_valid_steps(n_size, cache_step, s, nsize2):
            scrable_steps.append(s)
            cache_step = s

    scp = go(scrable_steps, 0, n_size=n_size)
    return scrable_steps, scp

def lcm(cp: CyclicPermutation) -> int:
    return math.lcm(*[len(g) for g in cp.groups])

In [9]:
for _ in range(1024):
    try:
        step, cp = sramble(3)
        inv = cp.inv()
        sol_step = step[-2::-1]
        sol_step.append(0)
        if math.gcd(*[len(g) for g in inv.groups]) > 1 and len(inv.groups) > 1:
            print(sol_step, step[-1], go(sol_step, step[-1], check_valid=False), inv, sep="\n")
            print()
    except:
        pass

[4, 7, 8, 5, 6, 0]
5
CyclicPermutation([[5, 0, 6], [8, 7, 4]])
CyclicPermutation([[0, 6, 5], [4, 8, 7]])

[4, 1, 2, 5, 8, 0]
5
CyclicPermutation([[5, 0, 8], [2, 1, 4]])
CyclicPermutation([[0, 8, 5], [4, 2, 1]])

[4, 5, 8, 0, 8, 7, 4, 5, 8, 0]
1
CyclicPermutation([[1, 0, 8, 4], [5, 7]])
CyclicPermutation([[0, 8, 4, 1], [5, 7]])

[4, 7, 8, 5, 6, 0]
5
CyclicPermutation([[5, 0, 6], [8, 7, 4]])
CyclicPermutation([[0, 6, 5], [4, 8, 7]])

[4, 1, 4, 1, 2, 5, 8, 0]
5
CyclicPermutation([[5, 0, 8], [2, 1, 4]])
CyclicPermutation([[0, 8, 5], [4, 2, 1]])

[2, 3, 6, 0, 6, 5, 8, 0]
5
CyclicPermutation([[5, 0, 8], [6, 3, 2]])
CyclicPermutation([[0, 8, 5], [2, 6, 3]])

[6, 3, 2, 5, 8, 0]
5
CyclicPermutation([[5, 0, 8], [2, 3, 6]])
CyclicPermutation([[0, 8, 5], [6, 2, 3]])



## Remember this

In [10]:
x = 5
y = 6
A = [7, 8]
B = [11, 12]
C = [15, 16]
go([*A, y, *B, x, *C, y],x, check_valid=False)

CyclicPermutation([[5, 6, 8, 7, 12, 11, 16, 15]])

In [ ]:
# 6, 5, 4, 1, 2, 5, 6, 0, 8, 5, 6, 0
table = [
        [4,1,3],
        [2,8,0],
        [7,6,5]
    ] - > table = [
        [4,1,3],
        [2,0, 8],
        [7,6,5]
    ] -> table = [
        [1,2,3],
        [4,0,8],
        [7,6,5]
    ] -> table = [
        [1,2,3],
        [4,8,0],
        [7,6,5]
    ] -> table = [
        [1,2,3],
        [4,8,5],
        [7,6,0]
    ] -> table = [
        [1,2,3],
        [4,5,6],
        [7,8,0]
    ]

SyntaxError: invalid syntax (1326912366.py, line 6)

In [57]:
table = [
        [4,1,3],
        [2,8,0],
        [7,6,5]
    ]
cp_prob, start = build_cp_from_tables(table)
cp_prob.inv() == go([5, 4, 1, 2, 5, 6, 0, 8, 5,6,0], 6, n_size=3)

True

In [58]:
go([5, 4, 1, 2, 5, 6], 6, n_size=3)

CyclicPermutation([[6], [5], [2, 1, 4]])

In [157]:
table =  [
        [10, 3, 6, 4],
        [ 1, 5, 8, 0],
        [ 2,13, 7,15],
        [14, 9,12,11]
    ]
cp, start = build_cp_from_tables(table)
inv = cp.inv()
start, inv

(8,
 CyclicPermutation([[1, 10, 13, 14, 9, 2, 3, 6, 5], [4], [7, 8, 0, 11], [12, 15]]))

In [ ]:
# 1 2 3 4
# 5 6 7 8
# 9 10 11 12
# 13 14 15 0

def solve_lcm(cp: CyclicPermutation, start, nsize):
    res = []
    last = start
    nsize2 = nsize ** 2
    while True:
        minlcm = 999
        minnex = None
        res.append(-1)
        for nex in _yield_valid_steps(nsize, last, nsize2):
            if nex != last:
                res[-1] = nex
                # print(res)
                lll = lcm(cp - go(res, start, n_size=nsize))

                if minnex is None or lll < minlcm:
                    minnex = nex
                    minlcm = lll
        

        res[-1] = minnex
        last = minnex
        print(cp - go(res, start, n_size=nsize), minlcm, minnex)
        
        if cp == go(res, start, n_size= nsize):
            return res

solve_lcm(inv, start, nsize=len(table))


CyclicPermutation([[1, 10, 13, 14, 9, 2, 3, 6, 5], [4], [7], [8, 0, 11], [12, 15]]) 18 7
CyclicPermutation([[1, 10, 13, 14, 9, 2, 3, 8, 0, 11, 6, 5], [4], [7], [12, 15]]) 12 6
CyclicPermutation([[1, 8, 0, 11, 6, 5], [10, 13, 14, 9, 2, 3], [4], [7], [12, 15]]) 6 10
CyclicPermutation([[1, 9, 2, 3, 10, 13, 14, 8, 0, 11, 6, 5], [4], [7], [12, 15]]) 12 9
CyclicPermutation([[1, 8, 0, 11, 6, 5], [10, 13, 14, 9, 2, 3], [4], [7], [12, 15]]) 6 10
CyclicPermutation([[1, 9, 2, 3, 10, 13, 14, 8, 0, 11, 6, 5], [4], [7], [12, 15]]) 12 9
CyclicPermutation([[1, 8, 0, 11, 6, 5], [10, 13, 14, 9, 2, 3], [4], [7], [12, 15]]) 6 10
CyclicPermutation([[1, 9, 2, 3, 10, 13, 14, 8, 0, 11, 6, 5], [4], [7], [12, 15]]) 12 9
CyclicPermutation([[1, 8, 0, 11, 6, 5], [10, 13, 14, 9, 2, 3], [4], [7], [12, 15]]) 6 10
CyclicPermutation([[1, 9, 2, 3, 10, 13, 14, 8, 0, 11, 6, 5], [4], [7], [12, 15]]) 12 9
CyclicPermutation([[1, 8, 0, 11, 6, 5], [10, 13, 14, 9, 2, 3], [4], [7], [12, 15]]) 6 10
CyclicPermutation([[1, 9, 2, 3,

In [107]:
inv - go([12, 11, 7, 6, 5, 1 ,2, 3, 7, 8, 12, 11, 15, 14, 13, 9], start, n_size=len(table)) 

CyclicPermutation([[1, 10, 9, 3, 5, 2, 6], [13], [14, 8, 0, 15, 7, 11, 12], [4]])